# Session B: train, evaluate and visualise the mitosis detector (GPU)

Uses the dataset built by Session A (`midog_yolo.zip`).

**Before running**
1. Right panel, **Add Input**, choose **Your Work** (Notebooks), add the Session A notebook.
2. *Session options*: **Accelerator = GPU P100** (or GPU T4 x2), **Internet = On**.
3. **Save Version**, then **Save & Run All (Commit)**. Training is capped at `HOURS` so your GPU quota is safe.
4. When done, the *Output* tab has `results/` (metrics, plots, overviews, `RESULTS.md`) and `runs/` (weights).

In [ ]:
# Settings
EPOCHS = 40        # upper limit; early stopping usually ends sooner
HOURS  = 1.0       # hard cap on training time (overrides EPOCHS if reached first)
MODEL  = 'yolov8s.pt'
BATCH  = 16        # use 8 if you get a CUDA out-of-memory error
IMGSZ  = 640
WORK   = '/kaggle/working'
DATA   = '/tmp/midog_yolo'
CODE   = f'{WORK}/code'

In [ ]:
# Locate Session A's output and unpack it to local disk
import glob, os, zipfile, shutil, json
from pathlib import Path

if not Path(f'{DATA}/data.yaml').exists():
    zips = glob.glob('/kaggle/input/**/midog_yolo.zip', recursive=True)
    dirs = [Path(p).parent for p in glob.glob('/kaggle/input/**/midog_yolo/data.yaml', recursive=True)]
    if zips:
        print('Unzipping', zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall('/tmp')
    elif dirs:
        print('Copying', dirs[0])
        shutil.copytree(dirs[0], DATA)
    else:
        raise FileNotFoundError('Session A output not found. Use "Add Input" and add the Session A notebook.')

# point data.yaml at the unpacked location
yaml = Path(f'{DATA}/data.yaml').read_text().splitlines()
yaml = [f'path: {DATA}' if l.startswith('path:') else l for l in yaml]
Path(f'{DATA}/data.yaml').write_text('\n'.join(yaml) + '\n')
print(Path(f'{DATA}/data.yaml').read_text())
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DATA}/images/{s}')), 'tiles,', len(os.listdir(f'{DATA}/full/{s}')), 'full images')

In [ ]:
!pip install -q ultralytics tifffile imagecodecs
!nvidia-smi --query-gpu=name,memory.total --format=csv
os.makedirs(CODE, exist_ok=True)

In [ ]:
%%writefile /kaggle/working/code/tiled_inference.py
"""
Tiled (sliding-window) inference on large pathology images.

A YOLO model trained on 640 x 640 tiles cannot see a ~6,600 x 4,900 px image
at once without shrinking cells to a few pixels. Instead we:
  1. cut the image into overlapping tiles,
  2. skip background tiles (mostly white glass),
  3. run the detector on tiles in batches,
  4. shift tile coordinates back to the full image,
  5. merge duplicate detections from overlapping tiles (center-distance NMS),
  6. report the mitotic count, a density heatmap and an annotated overview.

Usage (single image or folder)
  python tiled_inference.py --weights runs/midog/midog_yolov8s/weights/best.pt \
      --source datasets/midog_yolo/full/test --conf 0.4 --out results/inference
"""

import argparse
import csv
import json
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = None


def read_rgb(path):
    path = Path(path)
    if path.suffix.lower() in (".tif", ".tiff"):
        try:
            import tifffile
            arr = tifffile.imread(str(path))
            if arr.ndim == 2:
                arr = np.stack([arr] * 3, -1)
            if arr.shape[0] in (3, 4) and arr.shape[-1] not in (3, 4):
                arr = np.moveaxis(arr, 0, -1)
            return np.ascontiguousarray(arr[..., :3]).astype(np.uint8)
        except Exception:
            pass
    return np.array(Image.open(path).convert("RGB"))


def tile_starts(length, tile, stride):
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, stride))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


def center_nms(dets, radius):
    """dets: (N, 5) [x1, y1, x2, y2, score]. Keep the highest-scoring box and
    drop any other box whose center is within `radius` px (duplicates from
    overlapping tiles, or two boxes on the same cell)."""
    if len(dets) == 0:
        return dets
    dets = dets[np.argsort(-dets[:, 4])]
    centers = np.stack([(dets[:, 0] + dets[:, 2]) / 2, (dets[:, 1] + dets[:, 3]) / 2], 1)
    keep, removed = [], np.zeros(len(dets), bool)
    for i in range(len(dets)):
        if removed[i]:
            continue
        keep.append(i)
        d = np.linalg.norm(centers - centers[i], axis=1)
        removed |= d < radius
    return dets[keep]


def predict_large_image(model, arr, tile=640, overlap=64, conf=0.05, batch=16,
                        nms_radius=25, device=None, bg_thresh=225):
    """Returns (N, 5) array of detections in full-image pixel coordinates."""
    h, w = arr.shape[:2]
    stride = tile - overlap
    coords, crops, all_dets = [], [], []

    def flush():
        if not crops:
            return
        results = model.predict(crops, imgsz=tile, conf=conf, device=device, verbose=False)
        for (x0, y0), r in zip(coords, results):
            b = r.boxes
            if len(b) == 0:
                continue
            xyxy = b.xyxy.cpu().numpy()
            sc = b.conf.cpu().numpy()[:, None]
            xyxy[:, [0, 2]] += x0
            xyxy[:, [1, 3]] += y0
            all_dets.append(np.hstack([xyxy, sc]))
        coords.clear()
        crops.clear()

    n_tiles = n_skipped = 0
    for y0 in tile_starts(h, tile, stride):
        for x0 in tile_starts(w, tile, stride):
            crop = arr[y0:y0 + tile, x0:x0 + tile]
            n_tiles += 1
            if crop.mean() > bg_thresh:
                n_skipped += 1
                continue
            if crop.shape[:2] != (tile, tile):
                pad = np.full((tile, tile, 3), 255, np.uint8)
                pad[:crop.shape[0], :crop.shape[1]] = crop
                crop = pad
            crops.append(np.ascontiguousarray(crop[..., ::-1]))  # Ultralytics expects BGR arrays
            coords.append((x0, y0))
            if len(crops) == batch:
                flush()
    flush()
    dets = np.vstack(all_dets) if all_dets else np.zeros((0, 5))
    return center_nms(dets, nms_radius), {"tiles": n_tiles, "background_skipped": n_skipped}


def density_heatmap(dets, shape, cell=512):
    h, w = shape[:2]
    grid = np.zeros((int(np.ceil(h / cell)), int(np.ceil(w / cell))))
    for x1, y1, x2, y2, _ in dets:
        cy = min(int((y1 + y2) / 2 // cell), grid.shape[0] - 1)
        cx = min(int((x1 + x2) / 2 // cell), grid.shape[1] - 1)
        grid[cy, cx] += 1
    return grid


def save_overview(arr, dets, path, max_side=2400, gt=None):
    """Downscaled image with predictions (red circles) and optional ground truth (green)."""
    h, w = arr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    img = Image.fromarray(arr).resize((int(w * s), int(h * s)))
    draw = ImageDraw.Draw(img)
    r = max(6, int(35 * s))
    for x1, y1, x2, y2 in (gt if gt is not None else []):
        cx, cy = (x1 + x2) / 2 * s, (y1 + y2) / 2 * s
        draw.ellipse([cx - r - 3, cy - r - 3, cx + r + 3, cy + r + 3], outline=(0, 200, 0), width=3)
    for x1, y1, x2, y2, sc in dets:
        cx, cy = (x1 + x2) / 2 * s, (y1 + y2) / 2 * s
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], outline=(230, 0, 0), width=3)
    img.save(path, quality=90)


def save_heatmap(grid, path, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(grid, cmap="magma", interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, label="mitoses per 512 x 512 px region")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--source", required=True, help="Image file or folder of TIFF/JPG/PNG")
    ap.add_argument("--out", default="results/inference")
    ap.add_argument("--conf", type=float, default=0.4, help="Use the best threshold found by evaluate_midog.py")
    ap.add_argument("--tile", type=int, default=640)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    from ultralytics import YOLO
    model = YOLO(args.weights)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    src = Path(args.source)
    exts = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}
    files = sorted(p for p in src.iterdir() if p.suffix.lower() in exts) if src.is_dir() else [src]

    summary = []
    for p in files:
        t0 = time.time()
        arr = read_rgb(p)
        dets, info = predict_large_image(model, arr, args.tile, args.overlap, args.conf,
                                         args.batch, device=args.device)
        dt = time.time() - t0
        mp = arr.shape[0] * arr.shape[1] / 1e6
        with open(out / f"{p.stem}_detections.csv", "w", newline="") as f:
            wr = csv.writer(f)
            wr.writerow(["x1", "y1", "x2", "y2", "confidence"])
            wr.writerows(np.round(dets, 3).tolist())
        save_overview(arr, dets, out / f"{p.stem}_overview.jpg")
        save_heatmap(density_heatmap(dets, arr.shape), out / f"{p.stem}_heatmap.png",
                     f"{p.name}: {len(dets)} mitotic figures")
        row = {"image": p.name, "width": arr.shape[1], "height": arr.shape[0],
               "mitotic_count": int(len(dets)), "seconds": round(dt, 2),
               "megapixels_per_second": round(mp / dt, 2), **info}
        summary.append(row)
        print(f"{p.name}: {len(dets)} mitoses | {arr.shape[1]}x{arr.shape[0]} px | "
              f"{dt:.1f}s ({mp / dt:.1f} MP/s) | {info['background_skipped']}/{info['tiles']} tiles skipped")

    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\nResults written to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/evaluate_midog.py
"""
Evaluate the mitosis detector the way the MIDOG challenge does: on FULL
images, with detection F1 score.

Steps
  1. Tiled inference on every full val and test image (low confidence, once).
  2. Match predictions to ground-truth mitoses: a prediction is a true positive
     if its center is within 30 px (about 7.5 micrometres, the MIDOG criterion)
     of an unmatched ground-truth mitosis. Greedy, highest confidence first.
  3. Pick the confidence threshold with the best F1 on VAL, then report TEST
     at that threshold (no tuning on test).
  4. Report overall and per-tumor-type precision / recall / F1, mitotic count
     error and count correlation, plus tile-level mAP from Ultralytics.

Usage
  python evaluate_midog.py --weights runs/midog/midog_yolov8s/weights/best.pt \
      --data-dir datasets/midog_yolo
"""

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np

from tiled_inference import predict_large_image, read_rgb, save_overview


def match(dets, gt_centers, radius):
    """Returns (tp, fp, fn) for detections already filtered by threshold."""
    if len(gt_centers) == 0:
        return 0, len(dets), 0
    if len(dets) == 0:
        return 0, 0, len(gt_centers)
    dets = dets[np.argsort(-dets[:, 4])]
    centers = np.stack([(dets[:, 0] + dets[:, 2]) / 2, (dets[:, 1] + dets[:, 3]) / 2], 1)
    used = np.zeros(len(gt_centers), bool)
    tp = 0
    for c in centers:
        d = np.linalg.norm(gt_centers - c, axis=1)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= radius:
            used[j] = True
            tp += 1
    return tp, len(dets) - tp, len(gt_centers) - tp


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def scores_at(cases, thr, radius):
    tot = np.zeros(3, int)
    per_type = defaultdict(lambda: np.zeros(3, int))
    for c in cases:
        d = c["dets"][c["dets"][:, 4] >= thr]
        m = np.array(match(d, c["gt"], radius))
        tot += m
        per_type[c["tumor"]] += m
    return tot, per_type


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data-dir", type=Path, default=Path("datasets/midog_yolo"))
    ap.add_argument("--ann", type=Path, default=None,
                    help="MIDOG++.json (default: <data-dir>/MIDOG++.json, copied there by Session A)")
    ap.add_argument("--out", type=Path, default=Path("results/evaluation"))
    ap.add_argument("--radius", type=float, default=30.0, help="Match distance in px (~7.5 um)")
    ap.add_argument("--tile", type=int, default=640)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default=None)
    ap.add_argument("--skip-map", action="store_true", help="Skip the tile-level mAP evaluation")
    args = ap.parse_args()

    from ultralytics import YOLO
    model = YOLO(args.weights)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "overviews").mkdir(exist_ok=True)

    ann = json.loads((args.ann or args.data_dir / "MIDOG++.json").read_text())
    gt_boxes = defaultdict(list)
    for a in ann["annotations"]:
        if a["category_id"] == 1:
            gt_boxes[a["image_id"]].append(a["bbox"])

    with open(args.data_dir / "split.csv") as f:
        split_rows = list(csv.DictReader(f))

    cases = {"val": [], "test": []}
    for row in split_rows:
        if row["split"] not in cases:
            continue
        path = args.data_dir / "full" / row["split"] / row["file_name"]
        if not path.exists():
            print(f"  ! missing {path}, skipped")
            continue
        arr = read_rgb(path)
        dets, _ = predict_large_image(model, arr, args.tile, args.overlap, conf=0.05,
                                      batch=args.batch, device=args.device)
        boxes = np.array(gt_boxes[int(row["image_id"])], float).reshape(-1, 4)
        gt = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2], 1) if len(boxes) else np.zeros((0, 2))
        cases[row["split"]].append({"name": row["file_name"], "tumor": row["tumor_type"],
                                    "dets": dets, "gt": gt, "gt_boxes": boxes, "arr_path": path})
        print(f"[{row['split']}] {row['file_name']} ({row['tumor_type']}): "
              f"{len(gt)} true mitoses, {int((dets[:, 4] >= 0.3).sum())} predicted at conf 0.3")

    # 1) threshold selection on val
    thresholds = np.round(np.arange(0.05, 0.96, 0.025), 3)
    sweep = []
    tune_on = "val" if cases["val"] else "test"
    for t in thresholds:
        tot, _ = scores_at(cases[tune_on], t, args.radius)
        sweep.append((t, *prf(*tot)))
    best_t, _, _, best_f1_val = max(sweep, key=lambda x: x[3])
    print(f"\nBest threshold on {tune_on}: {best_t:.3f} (F1 {best_f1_val:.3f})")

    # 2) report test at that threshold
    report = {"threshold": float(best_t), "tuned_on": tune_on, "match_radius_px": args.radius}
    for split in ("val", "test"):
        if not cases[split]:
            continue
        tot, per_type = scores_at(cases[split], best_t, args.radius)
        p, r, f = prf(*tot)
        true_counts = [len(c["gt"]) for c in cases[split]]
        pred_counts = [int((c["dets"][:, 4] >= best_t).sum()) for c in cases[split]]
        corr = float(np.corrcoef(true_counts, pred_counts)[0, 1]) if len(set(true_counts)) > 1 and len(set(pred_counts)) > 1 else None
        report[split] = {
            "images": len(cases[split]),
            "tp": int(tot[0]), "fp": int(tot[1]), "fn": int(tot[2]),
            "precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4),
            "mean_abs_count_error": round(float(np.mean(np.abs(np.array(true_counts) - np.array(pred_counts)))), 2),
            "count_correlation": None if corr is None else round(corr, 4),
            "per_tumor_type": {t: dict(zip(["precision", "recall", "f1"], [round(x, 4) for x in prf(*m)]),
                                       tp=int(m[0]), fp=int(m[1]), fn=int(m[2]))
                               for t, m in sorted(per_type.items())},
        }

    # 3) per-image counts + overview images for the test set
    with open(args.out / "per_image_counts.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["split", "image", "tumor_type", "true_count", "predicted_count"])
        for split in ("val", "test"):
            for c in cases[split]:
                wr.writerow([split, c["name"], c["tumor"], len(c["gt"]), int((c["dets"][:, 4] >= best_t).sum())])
    for c in cases["test"][:8]:
        save_overview(read_rgb(c["arr_path"]), c["dets"][c["dets"][:, 4] >= best_t],
                      args.out / "overviews" / f"{Path(c['name']).stem}.jpg", gt=c["gt_boxes"])

    # 4) threshold curve plot
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    sw = np.array(sweep)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, lab in [(1, "precision"), (2, "recall"), (3, "F1")]:
        ax.plot(sw[:, 0], sw[:, i], label=lab, lw=2)
    ax.axvline(best_t, ls="--", c="gray")
    ax.set_xlabel("confidence threshold"); ax.set_ylabel("score"); ax.set_ylim(0, 1)
    ax.set_title(f"Full-image detection on {tune_on} (best F1 {best_f1_val:.3f} at {best_t:.2f})")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
    fig.savefig(args.out / "threshold_curve.png", dpi=150); plt.close(fig)

    # 5) standard tile-level mAP on the test tiles
    if not args.skip_map:
        m = model.val(data=str(args.data_dir / "data.yaml"), split="test", imgsz=args.tile,
                      batch=args.batch, device=args.device, plots=True, verbose=False,
                      project=str(args.out), name="tile_map", exist_ok=True)
        report["test_tiles_map"] = {"mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
                                    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4)}

    (args.out / "metrics.json").write_text(json.dumps(report, indent=2))
    print("\n" + json.dumps({k: v for k, v in report.items() if k != "val"}, indent=2))
    print(f"\nAll results in {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/train_midog.py
"""
Fine-tune YOLOv8 for mitotic figure detection on the tiled MIDOG++ dataset.

Augmentations are applied on the fly by Ultralytics (nothing extra is saved
to disk). They are tuned for histopathology:
  * vertical + horizontal flips: tissue has no "up", so both are always valid
  * strong HSV jitter: simulates stain and scanner colour variation, the main
    source of domain shift between labs
  * mild scale + mosaic; no shear/perspective (would distort cell shape)

Usage
  python train_midog.py --data datasets/midog_yolo/data.yaml
  python train_midog.py --data ... --epochs 40 --batch 16 --model yolov8s.pt
"""

import argparse
import time
from pathlib import Path

import torch
from ultralytics import YOLO


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--model", default="yolov8s.pt", help="Pretrained weights to start from")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16, help="Lower to 8 if you run out of GPU memory")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--patience", type=int, default=12)
    ap.add_argument("--name", default="midog_yolov8s")
    ap.add_argument("--device", default=None, help="0 for first GPU, cpu for CPU")
    ap.add_argument("--hours", type=float, default=None,
                    help="Hard time budget for training; overrides --epochs (safety net for limited GPU time)")
    ap.add_argument("--project", default="runs/midog")
    ap.add_argument("--cache", default="ram", help="ram / disk / none: caching decoded tiles speeds up epochs")
    args = ap.parse_args()

    device = args.device if args.device is not None else (0 if torch.cuda.is_available() else "cpu")
    if device == "cpu":
        print("WARNING: no CUDA GPU found, training on CPU will be very slow.")
    else:
        print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

    model = YOLO(args.model)
    t0 = time.time()
    extra = {"time": args.hours} if args.hours else {}
    model.train(
        **extra,
        data=args.data,
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        workers=args.workers,
        device=device,
        project=args.project,
        name=args.name,
        exist_ok=True,
        cache=False if args.cache == "none" else args.cache,
        optimizer="AdamW",
        lr0=0.001,
        cos_lr=True,
        warmup_epochs=2,
        amp=True,
        patience=args.patience,
        # histopathology augmentations
        fliplr=0.5,
        flipud=0.5,
        degrees=0.0,
        hsv_h=0.04,
        hsv_s=0.5,
        hsv_v=0.3,
        scale=0.2,
        translate=0.1,
        shear=0.0,
        perspective=0.0,
        mosaic=1.0,
        close_mosaic=5,
        mixup=0.0,
        plots=True,
        verbose=True,
    )
    print(f"\nTraining time: {(time.time() - t0) / 60:.1f} min")
    save_dir = Path(model.trainer.save_dir).resolve()
    best = save_dir / "weights" / "best.pt"
    Path("last_run.txt").write_text(str(save_dir))  # lets the notebook find the run reliably
    print(f"Run folder: {save_dir}")
    print(f"Best weights: {best}")


if __name__ == "__main__":
    main()

In [ ]:
# Train (about 30 to 60 minutes)
%cd {CODE}
!python train_midog.py --data {DATA}/data.yaml --model {MODEL} --epochs {EPOCHS} --hours {HOURS} --batch {BATCH} --imgsz {IMGSZ} --workers 4 --project {WORK}/runs --name midog_yolov8s
RUN = Path(open(f'{CODE}/last_run.txt').read().strip())
BEST = RUN / 'weights' / 'best.pt'
print('Best weights:', BEST, BEST.exists())

In [ ]:
# Evaluate on FULL val/test images (MIDOG-style F1) + tile-level mAP
!python evaluate_midog.py --weights {BEST} --data-dir {DATA} --out {WORK}/results/evaluation --batch {BATCH}

In [ ]:
# Tiled inference demo on test images: counts, heatmaps, speed
metrics = json.load(open(f'{WORK}/results/evaluation/metrics.json'))
THR = metrics['threshold']
!python tiled_inference.py --weights {BEST} --source {DATA}/full/test --conf {THR} --out {WORK}/results/inference --batch {BATCH}

In [ ]:
# Summary: the numbers for your README and resume
import csv
from IPython.display import Image as Show, display

m = json.load(open(f'{WORK}/results/evaluation/metrics.json'))
inf = json.load(open(f'{WORK}/results/inference/summary.json'))
man = json.load(open(f'{DATA}/manifest.json'))['splits']
t = m['test']
speed = sum(r['megapixels_per_second'] for r in inf) / len(inf)
skipped = sum(r['background_skipped'] for r in inf) / max(1, sum(r['tiles'] for r in inf))

lines = [
    '# MIDOG++ mitotic figure detection: results', '',
    f"Train tiles: {man['train']['tiles']} ({man['train']['mitoses_in_tiles']} mitoses) | "
    f"val images: {m['val']['images'] if 'val' in m else 0} | test images: {t['images']}", '',
    f"Confidence threshold (tuned on val): {m['threshold']:.3f}", '',
    '## Test set, full-image detection (match radius 30 px, about 7.5 um)', '',
    '| Precision | Recall | F1 | Mean abs. count error | Count correlation |',
    '|---|---|---|---|---|',
    f"| {t['precision']:.3f} | {t['recall']:.3f} | **{t['f1']:.3f}** | {t['mean_abs_count_error']} | {t['count_correlation']} |", '',
    '## Per tumor type (test)', '', '| Tumor type | Precision | Recall | F1 |', '|---|---|---|---|',
]
for k, v in t['per_tumor_type'].items():
    lines.append(f"| {k} | {v['precision']:.3f} | {v['recall']:.3f} | {v['f1']:.3f} |")
if 'test_tiles_map' in m:
    mm = m['test_tiles_map']
    lines += ['', f"Tile-level (test): mAP50 {mm['mAP50']:.3f}, mAP50-95 {mm['mAP50-95']:.3f}"]
lines += ['', f"Tiled inference speed: {speed:.1f} megapixels/s on {inf[0]['width']}x{inf[0]['height']} px images "
          f"(GPU), {skipped:.0%} background tiles skipped"]
Path(f'{WORK}/results/RESULTS.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))

for f in [RUN / 'results.png', f'{WORK}/results/evaluation/threshold_curve.png']:
    if Path(f).exists():
        display(Show(filename=str(f), width=800))
ov = sorted(glob.glob(f'{WORK}/results/evaluation/overviews/*.jpg'))
if ov:
    print('Green = ground truth, red = prediction:', Path(ov[0]).name)
    display(Show(filename=ov[0], width=900))

In [ ]:
# Keep the output small: remove the training cache files, keep weights + plots + results
for p in glob.glob(f'{WORK}/runs/**/*.cache', recursive=True):
    os.remove(p)
!du -sh {WORK}/runs {WORK}/results